# DDT Decoder 

This notebook provides a fixed implementation of the DDT code decoder, addressing the issues in the original implementation.

### Overview
The DDT (Diagnostic Data Trouble) Decoder is used to parse hex-encoded diagnostic codes from vehicle systems, specifically focusing on the BCM (Body Control Module) data. It extracts meaningful diagnostic information from the raw codes and provides human-readable interpretations using conversion tables.

### Key Components:

- decode_func.py - Main decoding function that interprets hex DDT strings
- Conversion tables in CSV format - For linear (L) and table (T) value interpretations
- DDT structure definitions - JSON files defining how to parse the binary data

### Implementation
First, let's import the required libraries:



In [0]:
# Import necessary libraries
from decode_func import decode_ddt
import json
import pandas as pd
import os

from pyspark.sql.functions import udf, col, to_timestamp
from pyspark.sql.types import StringType, IntegerType, FloatType, BooleanType
import pyspark.sql.functions as F

from pyspark.sql import SparkSession
from utils.decode_prepare import filter_valid_ddt_lengths, get_valid_rows_only, add_ddt_length_stats, apply_length_filtering, prepare_df_for_decoding
from helpers.alert_string_lengths import alrt_len
from utils.ddt_utils import optimized_flatten_json


# Initialize Spark session
spark = SparkSession.builder \
    .appName("DDT_Decoder") \
    .getOrCreate()



### Step 1: Define Paths to Conversion Files


In [0]:

fam_var = "WS"
my_var = "2025"
ddt_alerts = ['A001', 'A007']

families_to_process = fam_var

dir_path = f"/Workspace/Users/richard.sajdak@stellantis.com/dsci_12V_battery_ddt_decode/conversions/{fam_var}/{my_var}/"

DDT_file = {}

conv_L = f'conv_L_BCM_{fam_var}_{my_var}.csv'
conv_T = f'conv_T_BCM_{fam_var}_{my_var}.csv'

for ddt_alert in ddt_alerts:
    DDT_file[ddt_alert] = f'DDT_BCM_{ddt_alert}_{fam_var}_{my_var}.json'

save_path = f'gadp_scratch.rich.{fam_var.lower()}_sample'


### Step 2: Load and Prepare Vehicle Data

In [0]:
from pyspark.sql.functions import col, length

# Load data and clean it
df = spark.read.table("ods.adastream.area51_cc_na_tbm")

display(df)

Add "MODELYEAR", "BRANDNAME", "FAMILY"

In [0]:
# Load vehicle information
df_vehicle = spark.read.table("gadp_cdm.ods_cdm_vehicle.production")\
    .select("VIN", "MODELYEAR", "BRANDNAME", "FAMILY", "C_FAM")

# Join vehicle data with diagnostic data
df = df.join(df_vehicle, df["vin"] == df_vehicle["VIN"], "inner")\
    .select(df["vin"], df_vehicle["MODELYEAR"], df_vehicle["C_FAM"], df_vehicle["FAMILY"], "app_timestamp", "latitudegnss", "longitudegnss", "ignitiononcounter", col("ibs_tcsm_22a001").alias("BCM_A001"), col("ibs_tcsm_22a007").alias("BCM_A007"))

# Filter for specific vehicle family
df = df.filter((col("C_FAM") == fam_var))

# Then apply length validation and filtering
df_valid = prepare_df_for_decoding(df, fam_var, my_var, ddt_alerts, alrt_len)


#Limit to 100 rows for testing
df_valid = df_valid.limit(50)

display(df_valid)

### Step 3: Load Conversion Tables

In [0]:
from pyspark.sql.functions import when# Load conversion table

# Read the CSV file
conv_tbl_T = spark.read.csv(
    f"file://{dir_path}{conv_T}",
    header=True,
    inferSchema=True
)

# Ensure ID is integer type for consistent comparisons
if "ID" in conv_tbl_T.columns:
    conv_tbl_T = conv_tbl_T.withColumn(
        "ID",
        conv_tbl_T["ID"].cast("int")
    )
# Convert to pandas for easier filtering in the decode function
conv_tbl_pd_T = conv_tbl_T.toPandas()


#===============

# Load linear conversion table
conv_tbl_L = spark.read.csv(
    f'file://{dir_path}{conv_L}',
    header=True,
    inferSchema=True
)

if "ID" in conv_tbl_L.columns:
    conv_tbl_L = conv_tbl_L.withColumn(
        "ID",
        conv_tbl_L["ID"].cast("int")
    )

if "Decimal Places" in conv_tbl_L.columns:
    conv_tbl_L = conv_tbl_L.withColumn(
        "Decimal Places",
        when(
            col("Decimal Places").isin([float('inf'), float('-inf')]),
            0
        ).otherwise(col("Decimal Places"))
    )
    conv_tbl_L = conv_tbl_L.fillna({"Decimal Places": 0})
    conv_tbl_L = conv_tbl_L.withColumn(
        "Decimal Places",
        col("Decimal Places").cast("int")
    )

conv_tbl_pd_L = conv_tbl_L.toPandas()


# display(conv_tbl_pd_L)

In [0]:
# Register conversion table as a broadcast variable for efficiency
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

conv_tbl_pd_T_broadcast = sc.broadcast(conv_tbl_pd_T)
conv_tbl_pd_L_broadcast = sc.broadcast(conv_tbl_pd_L)

# Create a dictionary to cache conversion tables for performance
conversion_cache = {}

### Step 4: Define the Wrapper Function for DDT Decoding

In [0]:
# Create a serializable function to decode DDT strings that also includes vehicle information
def decode_ddt_wrapper(ddt_string, structure_name, family=None, model_year=None, ddt_type='A001'):
    """Wrapper for the decode_ddt function that works with UDFs and uses vehicle-specific conversions.
    
    Args:
        ddt_string: String containing the hex DDT code
        structure_name: Name of the structure to use for decoding
        family: Vehicle family (optional)
        model_year: Vehicle model year (optional)
        ddt_type: Type of DDT to decode (A001 or A007)
        
    Returns:
        JSON string with decoded data
    """
    if not ddt_string:
        return json.dumps({"error": "Empty DDT string"})
    
    try:
        # Get structure
        file_path = os.path.join(dir_path, DDT_file[ddt_type])
        with open(file_path, 'r') as file:
            json_structure = json.load(file)
            
        if not json_structure:
            return json.dumps({"error": f"Structure {structure_name} not found"})
        
        # Get the appropriate conversion tables
        if family and model_year:
            # This is where we would call the load_conversion_tables function
            conv_table_L = conv_tbl_pd_L_broadcast.value
            conv_table_T = conv_tbl_pd_T_broadcast.value
        else:
            # Fall back to the default broadcast tables if no family/year specified
            conv_table_L = conv_tbl_pd_L_broadcast.value
            conv_table_T = conv_tbl_pd_T_broadcast.value
        
        # Use the appropriate conversion tables
        result = decode_ddt(ddt_string, json_structure, conv_table_T, conv_table_L)
        
        # Add vehicle info to the result
        if family and model_year:
            result["vehicle_family"] = family
            result["model_year"] = model_year
        
        # Convert to JSON string for return
        return json.dumps(result)
    except Exception as e:
        return json.dumps({"error": str(e)})

### Step 5: Create UDFs for Decoding

In [0]:
# UDF for the A001 structure with vehicle info
decode_A001_vehicle_udf = udf(
    lambda ddt_string, family, model_year: decode_ddt_wrapper(
        ddt_string, "A001", family, model_year, ddt_type='A001'
    ), 
    StringType()
)

# UDF for the A007 structure with vehicle info
decode_A007_vehicle_udf = udf(
    lambda ddt_string, family, model_year: decode_ddt_wrapper(
        ddt_string, "A007", family, model_year, ddt_type='A007'
    ), 
    StringType()
)

### Step 6: Apply the UDFs to Decode the Data

In [0]:
# Apply the UDFs with vehicle information to decode both columns
df_decoded = df_valid.withColumn(
    "decoded_BCM_A001", 
    decode_A001_vehicle_udf(col("BCM_A001"), col("C_FAM"), col("MODELYEAR"))
).withColumn(
    "decoded_BCM_A007", 
    decode_A007_vehicle_udf(col("BCM_A007"), col("C_FAM"), col("MODELYEAR"))
)
     

# display(df_decoded).limit(100)

In [0]:
# display(df_decoded).limit(100)
# 
# df_decoded.write.mode("overwrite").saveAsTable("gadp_scratch.rich.test_decode")

In [0]:
# df_decoded = spark.read.table("gadp_scratch.rich.test_decode")



### Step 7: Apply JSON Flattening and Display Results

In [0]:
# Apply multi-family JSON flattening and display results for all families
print("Flattening JSON results for all families...")

# Define column types (these should work across families)
column_types_A001 = {
    '*Rbatt*': 'float',
    '*SOH_SUL*': 'float', 
    '*SOH_SUL_Accuracy*': 'string',
    '*Q_Received*': 'float',
    '*Q_Released*': 'float',
    '*SOC*': 'float',
    '*SOC_Accuracy*': 'string', 
    '*Ibatt*': 'float',
    '*Vbatt*': 'float',
    '*T_BATT*': 'float',
    '*voltage*': 'float'
}

column_types_A007 = {
    '*Rbatt*': 'float',
    '*SOH_SUL*': 'float',
    '*SOH_SUL_Accuracy*': 'string',
    '*Q_Received*': 'float', 
    '*Q_Released*': 'float',
    '*SOC*': 'float',
    '*SOC_Accuracy*': 'string',
    '*Ibatt*': 'float',
    '*Vbatt*': 'float',
    '*T_BATT*': 'float',
    '*voltage*': 'float',
    '*switch_status*': 'string'
}

# Flatten both decoded columns
d1 = optimized_flatten_json(
    df_decoded,
    "A001",
    "decoded_BCM_A001",
    column_types=column_types_A001,
    fam_var=fam_var,
    my_var=my_var
)


# Flatten the A007 column on the already flattened dataframe
d1 = optimized_flatten_json(
    d1,
    "A007",
    "decoded_BCM_A007",
    column_types=column_types_A007,
    fam_var=fam_var,
    my_var=my_var
)

# Convert timestamp to proper format and order
d1 = d1.withColumn(
    'app_timestamp', 
    to_timestamp(col('app_timestamp'), 'yyyy-MM-dd HH:mm:ss')
).orderBy('app_timestamp', ascending=True)

# Display the results
# display(d1)

In [0]:
display(d1)

## Write to output. 
_Modify path as needed_

In [0]:

# Uncomment to save:
# d1.write.mode('overwrite').format("delta").option('overwriteSchema', 'true').option('enableColumnMapping', 'true').saveAsTable(save_path)

